# 🚀 Gandharva-Omni-7B: 1-Click Fine-Tuning & Quantization Engine
### 100% Free Fine-Tuning on Kaggle (NVIDIA T4 / P100 GPU) using Unsloth (QLoRA 4-bit)

**Tasks Trained into Single Model:**
1. `[MODE: PROMPT_DIRECTOR]` — 150-word audio engineering prompts for MusicGen
2. `[MODE: LYRICS_STUDIO]` — 26-line structured songs (Telugu, Hindi, Tamil, English) with chords
3. `[MODE: NIE_BLUEPRINT]` — 100% deterministic JSON album blueprints
4. `[MODE: MUSIC_DIRECTOR]` — Tempo (BPM), Key signature, and Stem arrangement blueprints
5. `[MODE: VOCAL_COACH]` — Vocal expression, singing guidance, and pitch tips

In [ ]:
# Step 1: Install High-Speed Unsloth Training Framework (Free & Open Source)
!pip install --no-deps unsloth "xformers" "trl<0.9.0" peft accelerate bitsandbytes
print("✅ Unsloth installed successfully!")

In [ ]:
# Step 2: Load Qwen2.5-7B Multilingual Base Model in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection (Float16 or Bfloat16)
load_in_4bit = True # 4-bit quantization reduces VRAM to only 5.5GB!

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Step 3: Add Fast LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ Model & LoRA Adapters Loaded and Ready!")

In [ ]:
# Step 4: Load Gandharva Multi-Task Dataset
from datasets import load_dataset

# Upload gandharva_omni_train.jsonl to your Kaggle Notebook or download from your repo
dataset = load_dataset("json", data_files="gandharva_omni_train.jsonl", split="train")
print(f"✅ Dataset Loaded: {len(dataset)} verified samples across all 5 studio tasks")

In [ ]:
# Step 5: Execute Training (~45 Minutes on Kaggle Free GPU)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        warmup_steps=15,
        max_steps=350, # ~45-50 mins for optimal convergence
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="gandharva_omni_weights",
    ),
)

trainer_stats = trainer.train()
print("🎉 Training Complete! Model Loss converged beautifully!")

In [ ]:
# Step 6: Test Inference on Your Trained Model Live in Notebook
FastLanguageModel.for_inference(model)

test_prompt = """<|im_start|>system
You are Gandharva-Omni AI Engine. You operate in 5 modes.<|im_end|>
<|im_start|>user
[MODE: LYRICS_STUDIO]
Write song lyrics: Topic: "విజయ యాత్ర" | Language: Telugu | Mood: Motivation | Genre: Mass Anthem<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=0.75)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Step 7: Export to 4-bit GGUF for 100% Offline / High-Speed Serving (4.2 GB file)
model.save_pretrained_gguf("gandharva_omni_7b_q4", tokenizer, quantization_method="q4_k_m")
print("✅ Model successfully quantized to 4-bit GGUF!")

# Step 8: Push to Free Hugging Face Hub
# Replace with your Hugging Face Token (https://huggingface.co/settings/tokens)
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
model.push_to_hub_gguf("Prasanthm4734f/gandharva-omni-7b-gguf", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
print("🚀 Model uploaded to Hugging Face Hub successfully!")